# Practical 05: Settings in a File, Proof That Runs Itself

**YAML configuration and automated testing with pytest**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L10 |
| **Course Outcome** | CO3 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 450 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. Move every chosen setting out of the code and into a YAML file.
2. Read that file from Python and let it control training.
3. Write automated checks that run without supervision.
4. Run the whole suite with one command and interpret its report.

## What this practical is about

Two habits are introduced today. They appear unrelated. Both concern the same
property: whether the project can be changed safely.

**Configuration.** The training code written so far contains values somebody
chose: a test size of 0.2, a seed of 42, four column names. They are presently
typed into the body of the code. Such a value is called a **magic number**, and
it creates two problems. Changing one requires editing code, which carries the
risk of breaking it. Establishing which settings produced a given result
requires reading every line.

The remedy is to hold settings in a separate file. This practical uses **YAML**,
a format designed to be read by people. Indentation carries the structure. There
are no brackets and no commas. Docker Compose, GitHub Actions and Kubernetes all
read YAML, so the format recurs for the rest of the course.

**Testing.** A result confirmed by reading output on screen is confirmed only
for the person who was watching. An automated test records the same judgement in
a form that can be re-executed by anyone, at any time, without supervision. The
exit code it returns is what a pipeline reads in P11 to decide whether a change
may proceed.

## Before you start

P04 should be complete. A module can be written and imported, and the meaning of
a function returning a value is assumed.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite rather than a fault in the code.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell. Nothing is damaged by doing so.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'yaml', 'pytest', 'matplotlib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'yaml', 'pytest', 'matplotlib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent. This is a repository convenience, not a
recommendation to regenerate missing production data silently.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte for byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: observe the problem, settings buried in code

Below is training code in the style written so far. Seven values in it were
chosen by somebody: the four feature names, the target name, the test size and
the seed.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

orders = pd.read_csv(DATA)
X = orders[["distance_km", "prep_time_min", "traffic_level", "rain"]]
y = orders["delivery_min"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42)
mae = mean_absolute_error(y_te, LinearRegression().fit(X_tr, y_tr)
                          .predict(X_te))
print(f"MAE {mae:.2f}")

Changing the split to 70/30 requires editing that line. Establishing which
split produced yesterday's number requires reading the code as it stood
yesterday. Both costs are avoidable.

## Step 2: write the settings into a YAML file

Four YAML rules are sufficient for today.

- `key: value` declares one setting.
- Indentation with **spaces** groups settings together.
- `- item` declares a list entry.
- `#` begins a comment.

Tab characters must never appear in a YAML file. This is the most frequent
mistake with the format. The parser reports it as `found character '\t' that
cannot start any token`, with a line and column, so the message is precise once
it is recognised.

In [ ]:
config_yaml = """# Settings for the delivery-time model.
# Change values here, never in the code.

data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42

model:
  name: linear_regression
"""

(WORK / "config.yaml").write_text(config_yaml, encoding="utf-8")
print((WORK / "config.yaml").read_text(encoding="utf-8"))

## Step 3: read the file into Python

`yaml.safe_load` converts that text into ordinary Python dictionaries and lists.
Use `safe_load` and not `load`. The distinction matters: on current PyYAML,
`yaml.load(fh)` raises a `TypeError` because the loader argument is now
mandatory, while `yaml.unsafe_load` will construct arbitrary Python objects
named in the file and is therefore unsafe on any file received from elsewhere.

In [ ]:
import yaml

with open(WORK / "config.yaml", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)

print(type(config))
print()
print("features :", config["data"]["features"])
print("test size:", config["split"]["test_size"])
print("seed     :", config["split"]["seed"])

The result is a dictionary of dictionaries. The expression
`config["split"]["seed"]` reads in the same shape as the file, which is the
property YAML exists to provide.

## Step 4: train using only the configuration

The training is identical to Step 1. No chosen value now appears in the code.
Every setting is read from the file.

In [ ]:
def train_from_config(config, data_path):
    """Train and score, taking every setting from the config."""
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )
    model = LinearRegression().fit(X_tr, y_tr)
    return model, mean_absolute_error(y_te, model.predict(X_te))


model, mae = train_from_config(config, DATA)
print(f"MAE {mae:.2f} minutes")

Changing `test_size` to `0.3` is now an edit to a settings file rather than an
edit to code. No code review is required and no typographical error can break
training.

## Step 5: confirm that the configuration controls the behaviour

A setting that is read but never acted upon is indistinguishable from a comment.
The loop below changes one value and observes the consequence.

In [ ]:
for test_size in [0.2, 0.3, 0.4]:
    config["split"]["test_size"] = test_size
    _, mae = train_from_config(config, DATA)
    print(f"test_size {test_size} -> MAE {mae:.2f}")

config["split"]["test_size"] = 0.2   # put it back

## Step 6: plot what the settings change

Three printed numbers are easily forgotten. Plotting them, alongside the second
chosen value, makes the dependence visible.

The left panel holds code and data fixed and varies `test_size` across five
values. The right panel holds code and data fixed and varies the seed. Every bar
is an honest score. None of them is *the* MAE of this model.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

sizes = [0.1, 0.2, 0.3, 0.4, 0.5]
mae_by_size = []
for test_size in sizes:
    config["split"]["test_size"] = test_size
    mae_by_size.append(train_from_config(config, DATA)[1])
config["split"]["test_size"] = 0.2          # put it back

seeds = [1, 7, 42, 100, 2026]
mae_by_seed = []
for seed in seeds:
    config["split"]["seed"] = seed
    mae_by_seed.append(train_from_config(config, DATA)[1])
config["split"]["seed"] = 42                # put it back

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4),
                                  sharey=True)
bars = left.bar([str(s) for s in sizes], mae_by_size,
                color=["#2C5F79" if s == 0.2 else "#B9BFC7"
                       for s in sizes])
left.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
left.set_xlabel("split: test_size")
left.set_ylabel("MAE (minutes)")
left.set_title("change test_size in the file ...", fontsize=10)

bars = right.bar([str(s) for s in seeds], mae_by_seed,
                 color=["#2C5F79" if s == 42 else "#B9BFC7"
                        for s in seeds])
right.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
right.set_xlabel("split: seed")
right.set_title("... or change the seed: the score moves",
                fontsize=10)

for ax in (left, right):
    ax.set_ylim(0, 3)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The blue bars mark the settings presently recorded in `config.yaml`. A grey bar
that is higher or lower indicates nothing broken: the model is unchanged and
only the partition moved. This is precisely why chosen values belong in a file.
The statement "MAE 2.03" carries no information unless the `test_size` and the
seed that produced it can also be stated.

## Step 7: the assert statement

An `assert` declares that a condition must hold. If it holds, execution
continues with no output. If it does not, Python raises `AssertionError` and
stops.

In the cell below the first two statements produce nothing visible. The third is
caught deliberately so its message can be read.

In [ ]:
assert 2 + 2 == 4
assert "delivery".startswith("del")

try:
    assert 2 + 2 == 5, "arithmetic is broken"
except AssertionError as problem:
    print("AssertionError:", problem)

That is the entire mechanism. A test is a function containing assertions.

## Step 8: write a module worth testing

A test requires something to test. The module below provides two functions and
one deliberate rule: a delivery cannot cover zero distance.

In [ ]:
orders_py = '''"Small helpers for delivery orders."

TRAFFIC_LEVELS = (1, 2, 3)


def minutes_per_km(delivery_min, distance_km):
    "How many minutes each kilometre took."
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km


def is_valid_order(order):
    "True if this order is usable for training."
    if order.get("distance_km", 0) <= 0:
        return False
    if order.get("prep_time_min", 0) < 0:
        return False
    if order.get("traffic_level") not in TRAFFIC_LEVELS:
        return False
    if order.get("rain") not in (0, 1):
        return False
    return True
'''

(WORK / "orders.py").write_text(orders_py, encoding="utf-8")
print("wrote", WORK / "orders.py")

## Step 9: write the tests

pytest follows three conventions, all of them naming conventions.

- The file must be named `test_something.py`.
- Each test must be a function named `test_something`.
- Inside the function, a plain `assert` is used.

Note the second test. It confirms that an invalid input raises an error.
Establishing that a function fails correctly is as important as establishing
that it succeeds correctly, and it is the half more often omitted.

In [ ]:
test_py = '''"Tests for orders.py."

import pytest

from orders import is_valid_order, minutes_per_km


def test_minutes_per_km_simple():
    assert minutes_per_km(30, 10) == 3.0


def test_minutes_per_km_rejects_zero_distance():
    with pytest.raises(ValueError):
        minutes_per_km(30, 0)


def test_good_order_is_accepted():
    good = {"distance_km": 5.0, "prep_time_min": 20,
            "traffic_level": 2, "rain": 0}
    assert is_valid_order(good) is True


def test_bad_traffic_level_is_rejected():
    bad = {"distance_km": 5.0, "prep_time_min": 20,
           "traffic_level": 9, "rain": 0}
    assert is_valid_order(bad) is False
'''

(WORK / "test_orders.py").write_text(test_py, encoding="utf-8")
print("wrote", WORK / "test_orders.py")

## Step 10: run the suite with one command

`pytest` discovers every `test_*.py` file, runs every `test_*` function within
it, and reports. A dot denotes a pass and an `F` a failure. The `-v` flag prints
one line per test instead.

In [ ]:
import subprocess
import sys

def run_pytest(*extra):
    """Run pytest inside work/ and print its report."""
    done = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", *extra],
        cwd=WORK, capture_output=True, text=True,
    )
    print(done.stdout[-2500:])
    print("exit code:", done.returncode,
          "(0 means every test passed)")
    return done


run = run_pytest()

Four tests, four passes, exit code 0. That exit code is the value a pipeline
reads in P11 to decide whether a change is permitted to proceed. A person reads
the report; the pipeline reads only the number.

## Step 11: break the module deliberately

A suite that has never been observed to fail is a suite of unknown value. The
cell below introduces a fault, runs the suite, and then repairs the module.

In [ ]:
broken = (WORK / "orders.py").read_text(encoding="utf-8")
(WORK / "orders.py").write_text(
    broken.replace("return delivery_min / distance_km",
                   "return delivery_min * distance_km"),
    encoding="utf-8",
)

broken_run = run_pytest()

(WORK / "orders.py").write_text(broken, encoding="utf-8")  # repair
print()
print("module repaired")

Read the failure report. pytest states the line, the values it compared, and the
value it expected. That report is the reason tests are preferred to print
statements: the print statement records what happened, while the test records
what should have happened and the difference between the two.

## Step 12: plot the two runs as a pipeline would present them

In P11 a server will run this suite and display one row per test, coloured by
result. The server is not required to see that presentation. The `-v` output
already contains one line per test, so those lines can be parsed and drawn.

The left panel is the passing run from Step 10. The right panel is the broken
run from Step 11. The word on each bar carries the same information as its
colour, which keeps the figure readable without colour.

In [ ]:
%matplotlib inline
import re

import matplotlib.pyplot as plt

def test_results(run):
    """Read pytest's -v report into (test name, PASSED/FAILED)."""
    plain = re.sub(r"\x1b\[[0-9;]*m", "", run.stdout)  # drop colours
    found = re.findall(r"^(\S+::\S+) (PASSED|FAILED|ERROR)",
                       plain, flags=re.M)
    return [(name.split("::")[-1], status) for name, status in found]


def draw_test_board(run, ax, title):
    """One bar per test, blue for a pass, red for a failure."""
    results = test_results(run)[::-1]          # first test on top
    colours = ["#2C5F79" if s == "PASSED" else "#9E3B33"
               for _, s in results]
    ax.barh(range(len(results)), [1] * len(results),
            color=colours, height=0.7)
    for i, (name, status) in enumerate(results):
        ax.text(0.02, i, name, ha="left", va="center",
                color="white", fontsize=8.5)
        ax.text(0.98, i, status, ha="right", va="center",
                color="white", fontsize=8.5, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylim(-0.6, len(results) - 0.4)
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.set_title(f"{title}   (exit code {run.returncode})",
                 fontsize=10, loc="left")


fig, (left, right) = plt.subplots(1, 2, figsize=(10, 2.6))
draw_test_board(run, left, "the green run")
draw_test_board(broken_run, right, "after breaking orders.py")
plt.tight_layout()
plt.show()

One red bar, and the exit code moved from 0 to 1. The pipeline reads only the
exit code and a person reads only the red bar. Neither has to open a log to
establish that something broke. Retain `draw_test_board`; the runs produced in
Tasks T2 and T3 can be drawn with it.

## Step 13: test behaviour, not the exact score

Testing a model carries one characteristic trap: never assert `mae == 2.03`.
Such a test fails the day data is added, and its failure conveys nothing about
correctness. L24 demonstrates this twice, once on floating point precision and
once after a retrain.

Test **behaviour** that must hold instead. Rain must never make a delivery
faster. That property holds for any correct model of this problem, on any
dataset, at any seed.

In [ ]:
model_test_py = '''"Behaviour tests for the trained model."

import pandas as pd


def make_order(distance_km=5.0, prep_time_min=20,
               traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])


def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry


def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near


def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae
'''

(WORK / "test_model.py").write_text(model_test_py, encoding="utf-8")
print("wrote", WORK / "test_model.py")

Those tests require a trained model. Rather than train once inside every test,
pytest provides for shared preparation in a file named `conftest.py`. Each
prepared object is called a **fixture**, and a test requests one by naming it as
an argument.

## Step 14: share setup through conftest.py

`conftest.py` is a name pytest looks for automatically. It is never imported.
Any object decorated with `@pytest.fixture` within it can be requested by name
from any test in that folder.

In [ ]:
conftest_py = '''"Shared setup for every test in this folder."

from pathlib import Path

import pandas as pd
import pytest
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent


@pytest.fixture(scope="session")
def config():
    with open(HERE / "config.yaml", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


@pytest.fixture(scope="session")
def split(config):
    data_path = HERE.parent.parent / "data" / "delivery_times.csv"
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    return train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )


@pytest.fixture(scope="session")
def trained_model(split):
    X_tr, _, y_tr, _ = split
    return LinearRegression().fit(X_tr, y_tr)


@pytest.fixture(scope="session")
def model_mae(trained_model, split):
    _, X_te, _, y_te = split
    return mean_absolute_error(y_te, trained_model.predict(X_te))


@pytest.fixture(scope="session")
def baseline_mae(split):
    _, X_te, y_tr, y_te = split
    guess = [y_tr.mean()] * len(y_te)
    return mean_absolute_error(y_te, guess)
'''

(WORK / "conftest.py").write_text(conftest_py, encoding="utf-8")
run_pytest("-q")

Seven tests now run from one command: four concerning the helper functions and
three concerning the behaviour of the model.

## Step 15: plot what the behaviour tests promise

Each of the three model tests asserts one shape that must hold for any correct
delivery model on any data. The three shapes are drawn below from the model
trained in Step 4.

The left panel holds one order fixed and varies only rain. The middle panel
varies distance from 1 km to 12 km. The right panel compares the model's error
against a baseline that always predicts the training mean.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


def order(**changes):
    """One typical order, with some fields changed."""
    base = {"distance_km": 5.0, "prep_time_min": 20,
            "traffic_level": 2, "rain": 0}
    return pd.DataFrame([{**base, **changes}])


# the same split conftest.py builds, so the two errors are comparable
orders = pd.read_csv(DATA)
X_tr, X_te, y_tr, y_te = train_test_split(
    orders[config["data"]["features"]],
    orders[config["data"]["target"]],
    test_size=config["split"]["test_size"],
    random_state=config["split"]["seed"],
)
model_mae = mean_absolute_error(y_te, model.predict(X_te))
baseline_mae = mean_absolute_error(y_te, [y_tr.mean()] * len(y_te))

fig, (a, b, c) = plt.subplots(1, 3, figsize=(10, 3.2))

dry, wet = (float(model.predict(order(rain=r))[0]) for r in (0, 1))
bars = a.bar(["dry", "rain"], [dry, wet], color=["#B9BFC7", "#2C5F79"])
a.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
a.set_ylim(0, max(dry, wet) * 1.25)
a.set_ylabel("predicted minutes")
a.set_title("rain never makes it faster", fontsize=10)

kms = list(range(1, 13))
b.plot(kms, [float(model.predict(order(distance_km=k))[0]) for k in kms],
       marker="o", color="#2C5F79")
b.set_xticks([1, 3, 6, 9, 12])
b.set_xlabel("distance_km")
b.set_title("further is never faster", fontsize=10)

bars = c.bar(["guessing", "model"], [baseline_mae, model_mae],
             color=["#9E3B33", "#2C5F79"])
c.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
c.set_ylim(0, baseline_mae * 1.25)
c.set_ylabel("MAE (minutes)")
c.set_title("error is better than guessing", fontsize=10)

for ax in (a, b, c):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Rain produces the taller bar, the distance line is monotonic, and the model bar
sits well below the baseline. Adding a hundred further deliveries leaves every
one of those shapes intact, while the exact score in Step 6 moved with every
seed. Test shapes, not numbers. Task T3 adds a fourth shape: heavier traffic is
never faster.

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It prints a
table stating which tasks are correct. It may be run as often as required and
it never alters the notebook.

## Task T1: add a setting, and use it

Add a new section to the configuration named `training`, containing one setting:
`max_rows`, set to `300`.

Then write a function `train_limited(config, data_path)` that trains on only the
first `max_rows` observations of the dataset and returns the MAE.

Store the result of calling it in `T1_mae`.

> **Hint.** `config` is already a dictionary, so a section can be added
> directly: `config["training"] = {"max_rows": 300}`. Inside the function, read
> the CSV and then reduce it with
> `orders.head(config["training"]["max_rows"])` before splitting.

In [ ]:
# TODO: add a "training" section with max_rows set to 300
config["training"] = None


def train_limited(config, data_path):
    # TODO: read the data, keep only the first max_rows orders,
    #       split and train exactly as train_from_config does,
    #       and return the MAE.
    return None


T1_mae = train_limited(config, DATA)
print("MAE on the first 300 orders:", T1_mae)

## Task T2: write a test of your own

Add a new test file `work/test_speed.py` containing two tests for
`minutes_per_km` from `orders.py`.

1. `test_slow_delivery`: 60 minutes over 6 km must give 10.0.
2. `test_negative_distance_raises`: a distance of `-3` must raise `ValueError`.

Then run the whole suite and store the completed process in `T2_run`.

> **Hint.** Follow the shape of `test_orders.py` from Step 9. The second test
> requires `with pytest.raises(ValueError):` around the call. Run the suite with
> `T2_run = run_pytest("-q")`.

In [ ]:
speed_test_py = '''"My own tests for minutes_per_km."

import pytest

from orders import minutes_per_km


def test_slow_delivery():
    # TODO: assert that 60 minutes over 6 km is 10.0
    pass


def test_negative_distance_raises():
    # TODO: assert that a distance of -3 raises ValueError
    pass
'''

(WORK / "test_speed.py").write_text(speed_test_py, encoding="utf-8")

# TODO: run the test suite
T2_run = None

## Task T3: a behaviour test for the model

Add one further test to `work/test_model.py`, by appending rather than
replacing, named `test_heavier_traffic_is_never_faster`.

Using the `trained_model` fixture, it must assert that an order in traffic
level 3 is predicted to take at least as long as the same order in traffic
level 1.

Run the suite again and store the process in `T3_run`.

> **Hint.** The file already provides a `make_order(...)` helper, so
> `make_order(traffic_level=1)` and `make_order(traffic_level=3)` give the two
> orders. Open the file with `open(..., "a", encoding="utf-8")` to append. The
> test function must take `trained_model` as its argument.

In [ ]:
extra_test = '''

def test_heavier_traffic_is_never_faster(trained_model):
    # TODO: predict the same order at traffic level 1 and 3,
    #       then assert level 3 is not faster.
    pass
'''

with open(WORK / "test_model.py", "a", encoding="utf-8") as fh:
    fh.write(extra_test)

# TODO: run the suite again
T3_run = None

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | config has a training section with max_rows 300', lambda: config['training']['max_rows'] == 300)
_check('T1 | T1_mae is a sensible error in minutes', lambda: 0 < float(T1_mae) < 15)
_check('T1 | train_limited really used only 300 rows', lambda: abs(float(train_limited({**config, 'training': {'max_rows': 300}}, DATA)) - float(T1_mae)) < 1e-9 and abs(float(train_limited({**config, 'training': {'max_rows': 600}}, DATA)) - float(T1_mae)) > 1e-9)
_check('T2 | test_speed.py exists and names both tests', lambda: 'def test_slow_delivery' in (WORK / 'test_speed.py').read_text(encoding='utf-8') and 'def test_negative_distance_raises' in (WORK / 'test_speed.py').read_text(encoding='utf-8'))
_check('T2 | both tests really assert something', lambda: 'assert' in (WORK / 'test_speed.py').read_text(encoding='utf-8') and 'pytest.raises' in (WORK / 'test_speed.py').read_text(encoding='utf-8'))
_check('T2 | the whole suite passes (exit code 0)', lambda: T2_run.returncode == 0)
_check('T2 | at least 9 tests were collected', lambda: any(str(n) + ' passed' in T2_run.stdout for n in range(9, 40)))
_check('T3 | the new test asserts something and uses the fixture', lambda: 'test_heavier_traffic_is_never_faster(trained_model)' in (WORK / 'test_model.py').read_text(encoding='utf-8') and (WORK / 'test_model.py').read_text(encoding='utf-8').rsplit('def test_heavier', 1)[1].count('assert') >= 1)
_check('T3 | the whole suite still passes', lambda: T3_run.returncode == 0)

print("==================================================================")
print("SELF-CHECK   Practical 05: Settings in a File, Proof That Runs Itself")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The files `work/config.yaml` and `work/test_orders.py`.
3. In a markdown cell, one sentence stating why `assert mae == 2.03` is a poor
   test.

Name the file `P05_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, every test passing | 3 |
| Task T1, a new setting added and used | 2 |
| Task T2, a test written by you that passes | 3 |
| Task T3, a behaviour test for the model | 2 |
| **Total** | **10** |

## Read more

- pytest, getting started: <https://docs.pytest.org/en/stable/getting-started.html>
- pytest, fixtures: <https://docs.pytest.org/en/stable/explanation/fixtures.html>
- PyYAML documentation: <https://pyyaml.org/wiki/PyYAMLDocumentation>
- YAML specification, indentation rules: <https://yaml.org/spec/1.2.2/>